# 5주차. 동적 페이지 크롤링 패턴

| 항목 | 내용 |
|---|---|
| 이름 | 이성민 |
| 학과 | 소프트웨어융합과 |
| 학년 | 2학년 |
| 학번 | 2151050 |

이 노트북은 수업 주차 흐름을 참고해 우리 방식으로 재구성한 실행형 설명 자료입니다. 외부 서비스 호출 없이 실행되도록 작은 샘플 데이터를 사용합니다.

## 렌더링 이후 HTML
동적 페이지는 브라우저가 JavaScript를 실행한 뒤에 목록이 채워질 수 있습니다. 여기서는 렌더링이 끝난 HTML 조각을 입력으로 가정합니다.

In [1]:
from html.parser import HTMLParser
import pandas as pd

rendered = '''
<li class="toon"><a data-id="1">데이터 탐정</a><span class="author">Lee</span><span class="score">9.81</span></li>
<li class="toon"><a data-id="2">API 연구소</a><span class="author">Kim</span><span class="score">9.70</span></li>
<li class="toon"><a data-id="3">크롤링 노트</a><span class="author">Park</span><span class="score">9.65</span></li>
'''

class ToonParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.rows = []
        self.current = None
        self.field = None

    def handle_starttag(self, tag, attrs):
        attrs = dict(attrs)
        if tag == "li":
            self.current = {}
        elif self.current is not None and tag == "a":
            self.field = "title"
            self.current["id"] = int(attrs["data-id"])
        elif self.current is not None and tag == "span":
            self.field = attrs.get("class")

    def handle_data(self, data):
        text = data.strip()
        if self.current is not None and self.field and text:
            self.current[self.field] = text

    def handle_endtag(self, tag):
        if tag == "li" and self.current is not None:
            self.rows.append(self.current)
            self.current = None
        self.field = None

parser = ToonParser()
parser.feed(rendered)
toons = pd.DataFrame(parser.rows)
toons["score"] = toons["score"].astype(float)
display(toons)

,id,title,author,score
0,1,데이터 탐정,Lee,9.81
1,2,API 연구소,Kim,9.70
2,3,크롤링 노트,Park,9.65


In [2]:
def wait_for_minimum_rows(df, minimum):
    if len(df) < minimum:
        raise ValueError("렌더링된 행이 아직 부족합니다.")
    return df

checked = wait_for_minimum_rows(toons, 3)
best = checked.sort_values("score", ascending=False).iloc[0]

assert best["title"] == "데이터 탐정"
assert checked["score"].between(0, 10).all()
best.to_dict()

{'id': 1, 'title': '데이터 탐정', 'author': 'Lee', 'score': 9.81}